**Build Results Fact**

1. Read silver results table

2. Read silver sprints table

3. Add new column session_type with values RACE OF SPRINT

4. UNION results and sprints

5. Derive additional columns

    is win -> Indicates that the driver own the race

    is podium -> Indicates that the driver scored a podium result (1, 2, 3)

    has_points -> Indicates that the driver has scored points

6. Write the transformed data to gold fact_session_results table

In [0]:
%run "/Workspace/Users/ganeshgadade157@gmail.com/Projects/Formula-f1/00.common/01.envioment_config"

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import col

# Table name for gold dim_drivers
table_name = f"{catlog_name}.{gold_schema}.fact_session_results"

# Step 1 - Read silver drivers and gold ref_nationality_region tables
df_results  = spark.read.table("dev.silver.results")\
    .withColumn('session_type', f.lit('race'))\
    .drop('race_name','race_date','source_file_name','ingestion_timestamp')

df_sprints  = spark.read.table("dev.silver.sprints")\
    .withColumn('session_type', f.lit('sprint'))\
    .drop('race_name','race_date','source_file_name','ingestion_timestamp')

# Step 2 - Join silver drivers and gold ref_nationality_region)'sprint')

df_union= df_results.union(df_sprints)


df_result_sprint = df_union.withColumn('is_win',f.col('finish_position')==1)\
                           .withColumn('is_podium',f.col('finish_position').between(1,3))\
                           .withColumn('has_points',f.col('points')>0)

# Step 3 -Writing the data into table
            
df_result_sprint.write.format('delta').mode('overwrite').saveAsTable(table_name)
   






In [0]:
display(df_result_sprint)